<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/hard/optimizers/adamw/adamw.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("adamw", ...)`


# Implement AdamW from Scratch

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google

---

### Problem Statement

AdamW (**Loshchilov & Hutter, 2017**) fixes a flaw in the usual way weight decay is bolted onto Adam. With **L2 regularization** the penalty is added to the *gradient* (`g ← g + w·θ`), so it passes through the adaptive denominator — weights with large gradients get regularized *less*. **AdamW decouples** the decay from the gradient entirely:

```
θ ← θ − lr·w·θ        (directly on the parameter)
```

It is the default optimizer for training transformers.

### Tasks

1. `MyAdamW` — Adam's update plus weight decay applied **directly to the parameters**, not through the gradient, following PyTorch's `torch.optim.Optimizer` interface.

### References

- AdamW paper: https://arxiv.org/abs/1711.05101

In [ ]:
import math
import torch
from torch.optim import Optimizer


## The AdamW update

AdamW fixes a flaw in the usual way weight decay is bolted onto Adam. With **L2 regularization** the penalty is added to the *gradient* (`g ← g + w·θ`), so it passes through the adaptive denominator — weights with large gradients get regularized *less*. **AdamW decouples** the decay from the gradient entirely:

```
θ ← θ − lr·w·θ                     (directly on the parameter)
θ ← θ − lr·m̂_t / (√v̂_t + ε)        (the usual Adam step)
```

Note the decay is multiplied by `lr`, so the effective decay rate is `lr * weight_decay` — and a parameter with a **zero gradient must still decay**. Apply it to the parameter *before* the Adam step (i.e. decay θ_{t−1}), as `torch.optim.AdamW` does.

**Conventions worth knowing in an interview:**
- `torch.optim.Adam(..., weight_decay=w)` is the *L2* version by default (it adds `w·θ` to the gradient); `AdamW` decouples it. (Recent PyTorch also accepts `Adam(..., decoupled_weight_decay=True)`, which is the same algorithm as `AdamW`.)
- The paper writes the decay as `η_t·λ·θ`, where `η_t` is the schedule multiplier, not the learning rate α. PyTorch (and most libraries) scale it by the full `lr` instead, so `weight_decay` values are not directly comparable between the two conventions.
- In practice biases and normalization gains are usually put in a param group with `weight_decay=0`.

In [ ]:
class MyAdamW(Optimizer):
    """
    AdamW optimizer — Adam with decoupled weight decay.

    Args:
        params:       iterable of parameters to optimize
        lr:           learning rate (default 1e-3)
        betas:        coefficients for the moment estimates (default (0.9, 0.999))
        eps:          term for numerical stability (default 1e-8)
        weight_decay: weight decay coefficient (default 1e-2)
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay: {weight_decay}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad

                if grad.is_sparse:
                    raise RuntimeError("MyAdamW does not support sparse gradients")

                state = self.state[p]

                # TODO: state initialization (same as Adam):
                #       state['step'] = 0, state['exp_avg'], state['exp_avg_sq']
                ...

                # TODO: increment the step counter

                # TODO: biased moment estimates + bias corrections (same as Adam)

                # TODO: decoupled weight decay, applied directly to the
                #       parameter:  p.mul_(1 - lr * weight_decay)
                #       (do NOT add weight_decay * p to the gradient)

                # TODO: the Adam update (same as Adam):
                #       p -= lr * m_hat / (sqrt(v_hat) + eps)

        return loss


## Validation

`MyAdamW` is compared step-for-step against `torch.optim.AdamW` on a quadratic bowl, and decoupled weight decay is checked directly: with a zero gradient a parameter must still shrink by `(1 − lr·weight_decay)`, which Adam + L2 regularization does not do.

Until your implementation is in place these tests will fail — that is expected.


In [ ]:
def test_adamw():
    """Compare MyAdamW with torch.optim.AdamW. Key: AdamW != Adam with wd."""
    print("Testing AdamW...", end=" ")

    torch.manual_seed(42)
    D = 64
    target = torch.linspace(-1, 1, D)

    x_ref = torch.zeros(D, requires_grad=True)
    opt_ref = torch.optim.AdamW([x_ref], lr=0.1, betas=(0.9, 0.999),
                                eps=1e-8, weight_decay=0.01)

    x_my = torch.zeros(D, requires_grad=True)
    opt_my = MyAdamW([x_my], lr=0.1, betas=(0.9, 0.999),
                     eps=1e-8, weight_decay=0.01)

    for _ in range(100):
        loss_ref = ((x_ref - target) ** 2).mean()
        opt_ref.zero_grad()
        loss_ref.backward()
        opt_ref.step()

        loss_my = ((x_my - target) ** 2).mean()
        opt_my.zero_grad()
        loss_my.backward()
        opt_my.step()

    diff = (x_ref - x_my).abs().max().item()
    if diff < 1e-5:
        print(f"PASS  (max parameter diff: {diff:.2e})")
    else:
        print(f"FAIL  (max parameter diff: {diff:.2e})")


test_adamw()

In [ ]:
def test_adamw_weight_decay():
    """Decoupled decay acts on the parameter itself, so it works even with a ZERO
    gradient. Adam + L2 (the decay added to the gradient) does something else."""
    print("Testing AdamW decoupled weight decay...", end=" ")

    torch.manual_seed(42)
    lr, wd = 0.1, 0.5
    p0 = torch.randn(8)

    # Your MyAdamW with a zero gradient: the moments stay zero, so the step is
    # pure decay, p * (1 - lr * wd)
    p = p0.clone().requires_grad_(True)
    opt = MyAdamW([p], lr=lr, weight_decay=wd)
    p.grad = torch.zeros_like(p)
    opt.step()
    expected = p0 * (1 - lr * wd)

    # Adam + L2 for comparison: wd * p becomes the gradient and goes through the
    # adaptive denominator, so its first step is -lr * sign(p) instead
    q = p0.clone().requires_grad_(True)
    opt_l2 = torch.optim.Adam([q], lr=lr)
    q.grad = wd * p0.clone()
    opt_l2.step()

    diff = (p.detach() - expected).abs().max().item()
    gap = (q.detach() - expected).abs().max().item()
    if diff < 1e-6:
        print(f"PASS  (zero gradient -> p * (1 - lr*wd); Adam + L2 would be off by {gap:.3f})")
    else:
        print(f"FAIL  (with a zero gradient the step is {diff:.2e} away from p * (1 - lr*wd))")


test_adamw_weight_decay()

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("adamw", MyAdamW=MyAdamW)